# Transcribe overlapping speakers on separate recording channels

A stereo call can contain two people speaking at the same time. If each microphone is on its own channel, Lhotse can keep the two hypotheses separate. This example creates a two-channel recording, runs a local recognizer on each annotated interval, then reads the predictions back from a cut manifest. Speaker IDs, reference text and timings stay available for evaluation.

Use a Python 3.12 CPU kernel. The one-time downloads are two public audio excerpts and a 2.19 GB model. The example model, [Oruk Fourier 2B](https://huggingface.co/oruk/oruk-fourier-2b), is for English and has [noncommercial terms](https://huggingface.co/oruk/oruk-fourier-2b#license). To use another recognizer, replace the `recognize` callback.

In [ ]:
%pip install -q lhotse==1.33.0 transcribe-cpp==0.4.2 huggingface-hub soundfile urllib3

### Build one recording with two independent audio tracks

Channel 0 carries an excerpt from Kennedy’s inaugural address. Channel 1 carries a LibriSpeech reading. Both start two seconds into the recording. The cut starts half a second earlier, so the first sample of speech is at 0.5 seconds in cut coordinates.

In [ ]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlopen

import numpy as np
import soundfile as sf
from lhotse import CutSet, Recording, SupervisionSegment
from lhotse.qa import validate
from lhotse.utils import fastcopy

workspace = TemporaryDirectory()
root = Path(workspace.name)
fixtures = [
    ("jfk", "https://raw.githubusercontent.com/ggml-org/whisper.cpp/b0a11594aec50892a02cd8d129eee2dfe93a8bb8/samples/jfk.wav",
     "59dfb9a4acb36fe2a2affc14bacbee2920ff435cb13cc314a08c13f66ba7860e"),
    ("libri", "https://media.githubusercontent.com/media/andrewleech/qwen3-asr-onnx/4aee2ae0c355c7bc33bd5c09befec9e9eabd4b50/tests/fixtures/librispeech_0.wav",
     "799f78ed4beb4de7ceae3a809262d4ce242394342ccd1d58cef7d49dbc2def46"),
]
tracks = []
for name, url, digest in fixtures:
    data = urlopen(url, timeout=30).read()
    assert sha256(data).hexdigest() == digest
    path = root / f"{name}.wav"
    path.write_bytes(data)
    track, rate = sf.read(path, dtype="float32")
    assert rate == 16000 and track.ndim == 1
    tracks.append(track)

silence = 2 * rate
longest = max(map(len, tracks))
stereo = np.zeros((silence + longest, 2), dtype=np.float32)
for ch, track in enumerate(tracks):
    stereo[silence : silence + len(track), ch] = track
sf.write(root / "call.wav", stereo, rate)
recording = Recording.from_file(root / "call.wav")

In [ ]:
refs = [
    "And so my fellow Americans ask not what your country can do for you ask what you can do for your country",
    "Mr Quilter is the apostle of the middle classes and we are glad to welcome his gospel",
]
annotations = [
    SupervisionSegment(id=name, recording_id=recording.id, channel=ch, start=0.5,
                       duration=len(tracks[ch]) / rate, speaker=name, text=refs[ch])
    for ch, (name, _, _) in enumerate(fixtures)
]
cut = fastcopy(recording.to_cut(), id="call-excerpt", start=1.5,
               duration=0.5 + longest / rate, supervisions=annotations)
validate(cut)
print([(s.channel, s.start, s.speaker) for s in cut.supervisions])

### Select an interval *and* its channel

Supervision starts are relative to the cut. `Recording.load_audio` takes an offset relative to the recording. Add the cut start once, then use `channels=` to pick the microphone. The example uses one channel per supervision. A shared annotation covering multiple channels can instead be transcribed once per channel. This step assumes an existing segmentation.

In [ ]:
def with_hypotheses(cut, recognize, rate):
    recording = cut.recording.resample(rate)
    rows = []
    for s in cut.supervisions:
        assert isinstance(s.channel, int)
        start, end = max(0, s.start), min(cut.duration, s.end)
        if end <= start:
            continue
        frames = recording.load_audio(
            channels=s.channel, offset=cut.start + start, duration=end - start
        )
        rows.append({"supervision_id": s.id, "channel": s.channel,
                     "start": start, "duration": end - start, "text": recognize(frames[0])})
    if "asr_hypotheses" in (cut.custom or {}):
        raise ValueError("A hypothesis field already exists.")
    return fastcopy(cut, custom={**(cut.custom or {}), "asr_hypotheses": rows})

In [ ]:
from huggingface_hub import hf_hub_download
from transcribe_cpp import Model

path = hf_hub_download("oruk/oruk-fourier-2b", "Fourier-2B-Q8_0.gguf",
                       revision="8bcce21695e4ba13569c48de6cedffcb8a7576f5")
with Model(path, backend="cpu") as asr:
    def recognize(audio):
        with asr.session(n_threads=4) as session:
            return session.run(audio, language="en", timestamps="none").text.strip()

    annotated = with_hypotheses(cut, recognize, asr.capabilities.native_sample_rate)
print([(row["channel"], row["text"]) for row in annotated.asr_hypotheses])

### Save predictions without changing the reference

Write a new cut manifest. After reopening it, use the supervision ID to join each hypothesis to the corresponding reference. The start and duration describe the supplied interval, not estimated word times. The example is a channel-mapping check, not an accuracy benchmark.

In [ ]:
import re

out = root / "predictions.jsonl.gz"
CutSet.from_cuts([annotated]).to_file(out)
restored = next(iter(CutSet.from_file(out)))
validate(restored)
assert restored.supervisions == cut.supervisions
reference = {s.id: s for s in cut.supervisions}
for row in restored.asr_hypotheses:
    s = reference[row["supervision_id"]]
    assert s.channel == row["channel"]
    words = lambda text: re.findall(r"[a-z]+", text.lower())
    assert words(row["text"]) == words(s.text)
    print(f'{s.speaker}, channel {s.channel}: {row["text"]}')